# Observing the jacopy v3 proof system

This notebook walks through the proof engine after the E.1–E.5 strengthening pass:

1. **Definitional expansion** — watching rules fire one by one
2. **A full proof trace** — the `ExpandAndSimplify` pipeline
3. **`prefer=` API** — one identity, three "best" proofs
4. **Search anatomy** — state-graph statistics, beam fallback, portfolio
5. **TheoremBook** — prove once, cite forever (efficient vs foundational)
6. **Symmetry-aware simplify** — cancellations that need no rules


In [ ]:
from jacopy.core.registry import PropertyRegistry
from jacopy.core.expr import Integer, Neg, Product, Sum, Symbol
from jacopy.algebra.derivation import Act
from jacopy.central.objects import vector_fields, functions, forms, connection, frame
from jacopy.central.tangent import lie_bracket, tangent_engine
from jacopy.proof import (
    prove, SearchStrategy, PortfolioStrategy, ExpandAndSimplify,
    Theorem, TheoremBook, cite, ExpansionEngine,
)

reg = PropertyRegistry()
X, Y, Z = vector_fields("X Y Z")
f, g = functions("f g", registry=reg)
eng = tangent_engine(registry=reg)


def show(chain, width=100):
    """Pretty-print a ProofChain step by step."""
    for i, s in enumerate(chain.steps, 1):
        tag = getattr(s, "provenance_tag", None)
        tag_txt = f" [{tag}]" if tag else ""
        print(f"step {i}: {s.rule}{tag_txt}")
        print(f"   before: {s.before._repr_inner()[:width]}")
        print(f"   after : {s.after._repr_inner()[:width]}")
        for j, child in enumerate(getattr(s, "children", []) or [], 1):
            print(f"      sub {j}: {child.rule}")
    print(f"-- {len(chain.steps)} steps --")


## 1. Definitional expansion

The engine holds only **definitional** rules (canonical definitions — the Phase 2
definition policy). Watch `[X, Y](f)` unfold to its commutator definition:

In [2]:
expr = lie_bracket(X, Y)(f)
print("start:", expr._repr_inner(), "\n")
expanded, steps = eng.expand(expr)
for s in steps:
    print("fired:", s.rule)
print("\nresult:", expanded._repr_inner())

start: [X,Y]_VF(f) 

fired: Lie bracket definition: [X, Y](f) = X(Y(f)) - Y(X(f))

result: (X(Y(f)) + (-Y(X(f))))


## 2. A full proof trace

The Leibniz property `[X, fY](g) = X(f)·Y(g) + f·[X,Y](g)` is a **theorem** proved
from the definition. `ExpandAndSimplify` forms the obstruction `lhs − rhs` and
drives it to `0`:

In [3]:
from jacopy.central.tangent import prove_leibniz_second_slot
chain = prove_leibniz_second_slot(X, f, Y, g, registry=reg)
show(chain)

step 1: Lie bracket definition: [X, Y](f) = X(Y(f)) - Y(X(f)) [axiom]
   before: [X,(f * Y)]_VF(g)
   after : (X((f * Y)(g)) + (-(f * Y)(X(g))))
step 2: Lie bracket definition: [X, Y](f) = X(Y(f)) - Y(X(f)) [axiom]
   before: [X,Y]_VF(g)
   after : (X(Y(g)) + (-Y(X(g))))
step 3: product-rule
   before: ((X((f * Y)(g)) + (-(f * Y)(X(g)))) + (-((X(f) * Y(g)) + (f * (X(Y(g)) + (-Y(X(g))))))))
   after : ((X(f(Y(g))) + (-f(Y(X(g))))) + (-((X(f) * Y(g)) + (f * (X(Y(g)) + (-Y(X(g))))))))
step 4: scalar action: s(x) = s*x  (s a degree-0 scalar) [axiom]
   before: f(Y(g))
   after : (f * Y(g))
step 5: scalar action: s(x) = s*x  (s a degree-0 scalar) [axiom]
   before: f(Y(X(g)))
   after : (f * Y(X(g)))
step 6: product-rule
   before: ((X((f * Y(g))) + (-(f * Y(X(g))))) + (-((X(f) * Y(g)) + (f * (X(Y(g)) + (-Y(X(g))))))))
   after : ((((X(f) * Y(g)) + (f * X(Y(g)))) + (-(f * Y(X(g))))) + (-((X(f) * Y(g)) + (f * (X(Y(g)) + (-Y(X(g))
step 7: simplify
   before: ((((X(f) * Y(g)) + (f * X(Y(g)))) 

## 3. One identity, three "best" proofs — the `prefer=` API

`prove(..., prefer=...)` switches from the deterministic pipeline to **proof search**:
it explores alternative rule orders over the memoized state graph, collects every
proof it finds, scores them, and returns the best one under your preference.

In [4]:
lhs = Act(lie_bracket(X, Y), f)
rhs = Neg(Act(lie_bracket(Y, X), f))

s = SearchStrategy(prefer="shortest")
s.prove(lhs, rhs, registry=reg, engine=eng)
st = s.last_stats
print(f"explored {st['states_explored']} states, "
      f"found {st['solutions_found']} distinct proofs, "
      f"lengths {st['solution_lengths']}")

explored 12 states, found 4 distinct proofs, lengths [3, 4, 4, 4]


### Every proof the search found, in full

Same identity, four genuinely different routes to `0`. Note how they differ:
which bracket gets expanded first, and whether `simplify` runs once at the end
or twice along the way.

In [5]:
def show_steps(steps, width=90):
    for i, st in enumerate(steps, 1):
        print(f"  step {i}: {st.rule}")
        print(f"     before: {st.before._repr_inner()[:width]}")
        print(f"     after : {st.after._repr_inner()[:width]}")

for k, sol in enumerate(s.last_solutions, 1):
    print(f"=== proof #{k} — {len(sol)} steps ===")
    show_steps(sol)
    print()

=== proof #1 — 3 steps ===
  step 1: Lie bracket definition: [X, Y](f) = X(Y(f)) - Y(X(f))
     before: ([X,Y]_VF(f) + (-(-[Y,X]_VF(f))))
     after : ((X(Y(f)) + (-Y(X(f)))) + (-(-[Y,X]_VF(f))))
  step 2: Lie bracket definition: [X, Y](f) = X(Y(f)) - Y(X(f))
     before: ((X(Y(f)) + (-Y(X(f)))) + (-(-[Y,X]_VF(f))))
     after : ((X(Y(f)) + (-Y(X(f)))) + (-(-(Y(X(f)) + (-X(Y(f)))))))
  step 3: simplify
     before: ((X(Y(f)) + (-Y(X(f)))) + (-(-(Y(X(f)) + (-X(Y(f)))))))
     after : 0

=== proof #2 — 4 steps ===
  step 1: Lie bracket definition: [X, Y](f) = X(Y(f)) - Y(X(f))
     before: ([X,Y]_VF(f) + (-(-[Y,X]_VF(f))))
     after : ((X(Y(f)) + (-Y(X(f)))) + (-(-[Y,X]_VF(f))))
  step 2: simplify
     before: ((X(Y(f)) + (-Y(X(f)))) + (-(-[Y,X]_VF(f))))
     after : (X(Y(f)) + (-Y(X(f))) + [Y,X]_VF(f))
  step 3: Lie bracket definition: [X, Y](f) = X(Y(f)) - Y(X(f))
     before: (X(Y(f)) + (-Y(X(f))) + [Y,X]_VF(f))
     after : (X(Y(f)) + (-Y(X(f))) + (Y(X(f)) + (-X(Y(f)))))
  step 4: s

### What each scorer picks

* `shortest` and `elementary` pick **proof #1** (3 steps: expand both brackets, one simplify).
* `readable` scores by the *peak intermediate expression size*: proof #1 holds both
  expanded brackets at once (bigger peak), while the 4-step proofs simplify in between,
  keeping every intermediate line smaller — so `readable` picks one of those.

In [6]:
from jacopy.proof.search import SCORERS, _node_count

for pref in ("shortest", "readable", "elementary"):
    scorer = SCORERS[pref]
    scores = [scorer(sol) for sol in s.last_solutions]
    best = scores.index(min(scores)) + 1
    print(f"prefer={pref:10s}: scores per proof {scores}  -> picks proof #{best}")

prefer=shortest  : scores per proof [(3,), (4,), (4,), (4,)]  -> picks proof #1
prefer=readable  : scores per proof [(27, 3), (24, 4), (24, 4), (25, 4)]  -> picks proof #2
prefer=elementary: scores per proof [(0, 3), (0, 4), (0, 4), (0, 4)]  -> picks proof #1


## 4. Search anatomy on a bigger identity — Jacobi

`[X,[Y,Z]](f) + [Y,[Z,X]](f) + [Z,[X,Y]](f) = 0` needs at least 11 moves
(9 bracket expansions + Leibniz + simplify), so it is a real stress test.
Three runs, three behaviours:

1. **deterministic** `ExpandAndSimplify` — one fixed path, 11 steps;
2. **exhaustive search** with a large state budget — explores ~10k states,
   finds **8 distinct proofs**, returns the best;
3. **small budget** — BFS window fills before depth 11, the **beam fallback**
   still closes the proof (longer, but found); a portfolio recovers the
   compact proof by falling back to the deterministic strategy.

In [7]:
jac = Sum(
    Act(lie_bracket(X, lie_bracket(Y, Z)), f),
    Act(lie_bracket(Y, lie_bracket(Z, X)), f),
    Act(lie_bracket(Z, lie_bracket(X, Y)), f),
)

det = ExpandAndSimplify().prove(jac, Integer(0), registry=reg, engine=eng)
print("1) ExpandAndSimplify (deterministic):", len(det.steps), "steps")

1) ExpandAndSimplify (deterministic): 11 steps


In [8]:
# 2) exhaustive search — takes a few seconds, worth watching
sj = SearchStrategy(prefer="shortest", max_states=20000, max_solutions=8)
chain = sj.prove(jac, Integer(0), registry=reg, engine=eng)
st = sj.last_stats
print(f"explored {st['states_explored']} states")
print(f"found {st['solutions_found']} distinct proofs, lengths {st['solution_lengths']}")
print(f"engine: {st['engine']}  (budget exhausted: {st['budget_exhausted']})")
print(f"chosen: {st['chosen_length']} steps")

explored 10189 states
found 8 distinct proofs, lengths [11, 12, 12, 12, 12, 12, 12, 12]
engine: bfs  (budget exhausted: False)
chosen: 11 steps


### The chosen 11-step proof, in full

In [9]:
show_steps(chain.steps)

  step 1: Lie bracket definition: [X, Y](f) = X(Y(f)) - Y(X(f))
     before: (([X,[Y,Z]_VF]_VF(f) + [Y,[Z,X]_VF]_VF(f) + [Z,[X,Y]_VF]_VF(f)) + (-0))
     after : (((X([Y,Z]_VF(f)) + (-[Y,Z]_VF(X(f)))) + [Y,[Z,X]_VF]_VF(f) + [Z,[X,Y]_VF]_VF(f)) + (-0))
  step 2: Lie bracket definition: [X, Y](f) = X(Y(f)) - Y(X(f))
     before: (((X([Y,Z]_VF(f)) + (-[Y,Z]_VF(X(f)))) + [Y,[Z,X]_VF]_VF(f) + [Z,[X,Y]_VF]_VF(f)) + (-0))
     after : (((X((Y(Z(f)) + (-Z(Y(f))))) + (-[Y,Z]_VF(X(f)))) + [Y,[Z,X]_VF]_VF(f) + [Z,[X,Y]_VF]_VF(f
  step 3: Lie bracket definition: [X, Y](f) = X(Y(f)) - Y(X(f))
     before: (((X((Y(Z(f)) + (-Z(Y(f))))) + (-[Y,Z]_VF(X(f)))) + [Y,[Z,X]_VF]_VF(f) + [Z,[X,Y]_VF]_VF(f
     after : (((X((Y(Z(f)) + (-Z(Y(f))))) + (-(Y(Z(X(f))) + (-Z(Y(X(f))))))) + [Y,[Z,X]_VF]_VF(f) + [Z,
  step 4: Lie bracket definition: [X, Y](f) = X(Y(f)) - Y(X(f))
     before: (((X((Y(Z(f)) + (-Z(Y(f))))) + (-(Y(Z(X(f))) + (-Z(Y(X(f))))))) + [Y,[Z,X]_VF]_VF(f) + [Z,
     after : (((X((Y(Z(f)) + (-Z(Y(f)

### One of the 12-step alternatives, for contrast

Same theorem, different route: this proof pays one extra `simplify` mid-way
(smaller intermediate lines) instead of expanding everything first.

In [ ]:
# Pick an ALTERNATIVE solution when the search found one —
# step counts shift as the engine evolves, so we select from
# what was actually found instead of assuming a fixed length
# (2026-09-07 audit, finding 8).
shortest = min(sj.last_solutions, key=len)
alternatives = [s for s in sj.last_solutions if s is not shortest]
if alternatives:
    alt = min(alternatives, key=len)
    show_steps(alt)
else:
    print("the search found a single solution:", len(shortest), "steps")
    show_steps(shortest)


In [11]:
# 3) small budget: BFS window fills -> beam fallback still closes it
s_small = SearchStrategy(prefer="shortest", max_states=2000)
chain_beam = s_small.prove(jac, Integer(0), registry=reg, engine=eng)
print("small budget:", s_small.last_stats)

# ...and a portfolio recovers the compact proof deterministically:
pf = PortfolioStrategy([
    SearchStrategy(prefer="shortest", max_depth=4, max_states=50),
    ExpandAndSimplify(),
])
chain_pf = pf.prove(jac, Integer(0), registry=reg, engine=eng)
print("portfolio    :", len(chain_pf.steps), "steps (via deterministic fallback)")

small budget: {'prefer': 'shortest', 'states_explored': 2001, 'solutions_found': 0, 'solution_lengths': [], 'engine': 'beam', 'budget_exhausted': True, 'chosen_length': 19}
portfolio    : 11 steps (via deterministic fallback)


## 5. TheoremBook — prove once, cite forever

Antisymmetry is proved from the definition, registered as a `Theorem` (with the
equation and a generality tag), then **cited** into an engine as a derived rule.
Citation is the only way a theorem enters a proof — definitional engines never
carry theorems (no circular proofs).

In [12]:
from jacopy.central.tangent import prove_antisymmetry

book = TheoremBook()
book.add(Theorem(
    name="lie_antisymmetry",
    statement="[X, Y](f) = -[Y, X](f)",
    lhs=Act(lie_bracket(X, Y), f),
    rhs=Neg(Act(lie_bracket(Y, X), f)),
    proof=prove_antisymmetry(X, Y, f, registry=reg),
    generality="generic-function",
))
print(book)

TheoremBook(1 theorems: lie_antisymmetry)


In [13]:
# Efficient mode: the citation fires as ONE step tagged [theorem]
cited = cite(ExpansionEngine([]), book, "lie_antisymmetry")
out, steps = cited.expand(Act(lie_bracket(X, Y), f))
for s in steps:
    print("fired:", s.rule, f"[{s.provenance_tag}]")
print("result:", out._repr_inner())

fired: theorem lie_antisymmetry ⇒: [X, Y](f) = -[Y, X](f) [theorem]
result: (-[Y,X]_VF(f))


In [14]:
# Foundational mode: the SAME citation, but the stored proof
# is inlined under the step as sub-steps -- derivation down to axioms.
cited_f = cite(ExpansionEngine([], mode="foundational"), book, "lie_antisymmetry")
out, steps = cited_f.expand(Act(lie_bracket(X, Y), f))
step = steps[0]
print("step:", step.rule, f"[{step.provenance_tag}]")
for j, child in enumerate(step.children, 1):
    print(f"   sub {j}: {child.rule}")

step: theorem lie_antisymmetry ⇒: [X, Y](f) = -[Y, X](f) [theorem]
   sub 1: Lie bracket definition: [X, Y](f) = X(Y(f)) - Y(X(f))
   sub 2: Lie bracket definition: [X, Y](f) = X(Y(f)) - Y(X(f))
   sub 3: simplify


## 6. Symmetry-aware simplify

Alternating evaluations now carry a canonical form, so orientation-flipped terms
cancel with **no engine rule at all** — and sums re-factor when it makes them smaller.

In [15]:
from jacopy.algorithms.simplify import simplify
from jacopy.core.multi_eval import MultiEval

(w,) = forms("\u03c9", degree=2)

print("\u03c9(Y,X) + \u03c9(X,Y)  ->", simplify(Sum(MultiEval(w, Y, X), MultiEval(w, X, Y)))._repr_inner())
print("\u03c9(X,X)           ->", simplify(MultiEval(w, X, X))._repr_inner())

A, B = Symbol("A"), Symbol("B")
print("f\u00b7A + f\u00b7B        ->", simplify(Sum(Product(f, A), Product(f, B)))._repr_inner())

ω(Y,X) + ω(X,Y)  -> 0
ω(X,X)           -> 0
f·A + f·B        -> (f * (A + B))


---
**Where this goes next:** Phase 2.C derives the exterior derivative `d` from the Lie
bracket via the intrinsic (Palais) formula; `d² = 0` will be proved with this engine
and registered in the TheoremBook, and later proofs will simply cite it.